# Music Annotation Project V7.5R: Corrected Development-Reference Rebuild

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook creates a new V7.5-compatible development reference without changing or deleting legacy V7.5 runs. It is designed specifically for the V7.8.2 implementation-equivalence audit.

## Purpose

The legacy V7.5 cache contains Telea arrays that fail the outside-mask immutability gate. This rebuild:

1. reads the existing development clean pages, aligned observed pages and reviewed corrected masks;
2. creates the dilation-2 removal mask;
3. reconstructs Telea only inside that removal mask;
4. asserts exact byte equality outside the mask;
5. runs the same `geometry_t80`, routing and surrounding-evidence functions used by the rebuilt V7.8 builder;
6. writes a new timestamped V7.5 reference run with the cache and tables required by V7.8.2;
7. preserves all prior development runs unchanged.

This notebook does **not** use the independent V7.8 pages or outcomes.

## Output compatibility

It writes under:

```text
MusicAnnotationProject/reconstruction_geometry_benchmark_v7_5/<timestamp>_telea_immutable/
```

including:

```text
tables/cache_status.csv
tables/surrounding_area_evidence_by_component.csv
tables/surrounding_area_evidence_summary.csv
tables/surrounding_evidence_truth_diagnostics.csv
cache/*.npz
v7_5_rebuild_manifest.json
executive_summary.txt
```

After this passes, V7.7 must be regenerated from this corrected V7.5 reference before the full V7.8.2 audit can pass Stage D.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib


In [ ]:
import cv2, json, hashlib, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'
OBSERVED_DIR=ROOT/'aligned'
CORRECTED_MASK_DIR=ROOT/'masks'/'corrected'
OUTPUT_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')+'_telea_immutable'
OUT=OUTPUT_ROOT/RUN_NAME
TABLES=OUT/'tables';CACHE=OUT/'cache';REVIEW=OUT/'review';ROUTING=OUT/'routing_maps'
for d in [OUT,TABLES,CACHE,REVIEW,ROUTING]:d.mkdir(parents=True,exist_ok=True)

MASK_DILATION=2
CROP_MARGIN=48
MIN_COMPONENT_AREA=20
GEOMETRY_THRESHOLD=80
BAND_WIDTH=4
CONTEXT_RING_RADIUS=18
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
PAGE_ALLOWLIST=None
MAX_PAGES=None

def image_map(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def array_hash(a):
 a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def object_hash(v):return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def immutable(original,candidate,mask):
 out=original.copy();out[mask>0]=candidate[mask>0]
 if not np.array_equal(out[mask==0],original[mask==0]):raise RuntimeError('Outside-mask immutability violation')
 return out

def binary(mask):return np.where(mask>0,255,0).astype(np.uint8)
def prepare_mask(seed,dilation=MASK_DILATION):
 mask=binary(seed)
 if dilation>0:mask=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
 return mask

clean_map=image_map(CLEAN_DIR);observed_map=image_map(OBSERVED_DIR);mask_map=image_map(CORRECTED_MASK_DIR)
pages=sorted(set(clean_map)&set(observed_map)&set(mask_map))
if PAGE_ALLOWLIST is not None:pages=[p for p in pages if p in set(PAGE_ALLOWLIST)]
if MAX_PAGES is not None:pages=pages[:MAX_PAGES]
if not pages:raise RuntimeError('No matched development clean/aligned/corrected-mask pages found.')
print('Development pages:',len(pages));print('New reference run:',OUT)


## Corrected mask-only Telea


In [ ]:
def padded_components(mask):
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);h,w=mask.shape;boxes=[]
 for i in range(1,n):
  x,y,bw,bh,area=stats[i]
  if area>=8:boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
 changed=True
 while changed:
  changed=False;merged=[]
  while boxes:
   ax0,ay0,ax1,ay1=boxes.pop();rest=[]
   for bx0,by0,bx1,by1 in boxes:
    if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
     ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
    else:rest.append((bx0,by0,bx1,by1))
   boxes=rest;merged.append((ax0,ay0,ax1,ay1))
  boxes=merged
 return boxes

def telea_reconstruct(observed,mask):
 mask=binary(mask);candidate_result=observed.copy()
 for x0,y0,x1,y1 in padded_components(mask):
  im=observed[y0:y1,x0:x1];mk=mask[y0:y1,x0:x1]
  candidate=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA)
  local=candidate_result[y0:y1,x0:x1];local[mk>0]=candidate[mk>0];candidate_result[y0:y1,x0:x1]=local
 output=observed.copy();output[mask>0]=candidate_result[mask>0]
 drift=int(np.count_nonzero(output[mask==0]!=observed[mask==0]))
 if drift:raise RuntimeError(f'Telea changed {drift} outside-mask pixels')
 return output


## V7.5 geometry, routing and surrounding evidence


In [ ]:
def background_surface(observed,mask,sigma=45.0):
 visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32)
 num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma);surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma)
 return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)
def support_maps(observed,mask):
 dark=(observed<190)&(mask==0)
 horizontal=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
 vertical=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
 n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);compact=np.zeros_like(dark)
 for i in range(1,n):
  x,y,w,h,a=stats[i]
  if 3<=a<=650 and w<=40 and h<=40:compact[lab==i]=True
 k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8)
 return cv2.dilate(horizontal.astype(np.uint8),k)>0,cv2.dilate(vertical.astype(np.uint8),k)>0,cv2.dilate(compact.astype(np.uint8),k)>0

def geometry_t80(observed,mask,telea):
 paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255);horizontal,vertical,compact=support_maps(observed,mask);m=mask>0
 preserve=m&(horizontal|vertical|compact)&(observed<200)
 remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD)
 ambiguous=m&~preserve&~remove
 result=observed.copy();result[remove]=paper[remove];result[ambiguous]=telea[ambiguous];result[preserve]=observed[preserve]
 result=immutable(observed,result,mask)
 return result,{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'paper':paper,'normalised':normalised}

def surrounding_features(observed,component,routes):
 ys,xs=np.where(component);x0,x1=xs.min(),xs.max()+1;y0,y1=ys.min(),ys.max()+1;area=int(component.sum());w=x1-x0;h=y1-y0
 kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*CONTEXT_RING_RADIUS+1,2*CONTEXT_RING_RADIUS+1));ring=(cv2.dilate(component.astype(np.uint8),kernel)>0)&~component
 vals=observed[ring];edge=cv2.Canny(observed,70,170)>0;depth=cv2.distanceTransform(component.astype(np.uint8),cv2.DIST_L2,5)
 def f(k):return float(np.sum(routes[k]&component)/area)
 return {'w':w,'h':h,'area':area,'aspect_ratio':max(w,h)/max(min(w,h),1),'fill_ratio':area/max(w*h,1),'touches_page_border':bool(x0==0 or y0==0 or x1==observed.shape[1] or y1==observed.shape[0]),'context_pixels':int(ring.sum()),'context_edge_density':float(edge[ring].mean()) if ring.any() else 0.0,'context_dark_density':float((vals<190).mean()) if vals.size else 0.0,'context_mean_intensity':float(vals.mean()) if vals.size else 255.0,'context_intensity_std':float(vals.std()) if vals.size else 0.0,'maximum_mask_depth':float(depth[component].max()),'horizontal_fraction':f('horizontal'),'vertical_fraction':f('vertical'),'compact_fraction':f('compact'),'preserve_fraction':f('preserve'),'remove_fraction':f('remove'),'ambiguous_fraction':f('ambiguous'),'reconstruction_dependency_fraction':f('ambiguous'),'component_pixels':area}


## Post-hoc clean-truth diagnostics


In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def truth_diagnostics(clean,component,routes):
 ink=printed_ink(clean);paper=~ink;preserve=routes['preserve']&component;remove=routes['remove']&component;ambiguous=routes['ambiguous']&component
 return {'clean_print_fraction':float(np.sum(ink&component)/component.sum()),'clean_paper_fraction':float(np.sum(paper&component)/component.sum()),'preserved_clean_print_precision':float(np.sum(preserve&ink)/max(preserve.sum(),1)) if preserve.any() else np.nan,'removed_clean_paper_precision':float(np.sum(remove&paper)/max(remove.sum(),1)) if remove.any() else np.nan,'ambiguous_clean_print_fraction':float(np.sum(ambiguous&ink)/max(ambiguous.sum(),1)) if ambiguous.any() else np.nan}


## Build the corrected durable reference


In [ ]:
cache_rows=[];evidence_rows=[];truth_rows=[];integrity_rows=[];page_summary=[]
for index,page in enumerate(pages,1):
 clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0);seed=cv2.imread(str(mask_map[page]),0)
 if clean is None or observed is None or seed is None:raise RuntimeError(f'{page}: unreadable input')
 if clean.shape!=observed.shape or clean.shape!=seed.shape:raise RuntimeError(f'{page}: input shape mismatch')
 mask=prepare_mask(seed,MASK_DILATION);telea=telea_reconstruct(observed,mask);geometry,routes=geometry_t80(observed,mask,telea)
 telea_drift=int(np.count_nonzero(telea[mask==0]!=observed[mask==0]));geometry_drift=int(np.count_nonzero(geometry[mask==0]!=observed[mask==0]))
 if telea_drift or geometry_drift:raise RuntimeError(f'{page}: outside-mask drift telea={telea_drift}, geometry={geometry_drift}')
 cache_path=CACHE/f'{page}.npz'
 np.savez_compressed(cache_path,mask=mask,telea=telea,geometry_t80=geometry,**{f'g80_{k}':routes[k].astype(np.uint8) for k in ['preserve','remove','ambiguous','horizontal','vertical','compact']})
 cache_rows.append({'page':page,'cache':str(cache_path),'cache_sha256':sha256_file(cache_path),'mask_hash':array_hash(mask),'telea_hash':array_hash(telea),'geometry_t80_hash':array_hash(geometry),'status':'complete'})
 integrity_rows.append({'page':page,'observed_shape_ok':observed.shape==clean.shape,'mask_shape_ok':mask.shape==observed.shape,'telea_shape_ok':telea.shape==observed.shape,'telea_outside_mask_drift_pixels':telea_drift,'telea_outside_mask_immutable':telea_drift==0,'geometry_outside_mask_drift_pixels':geometry_drift,'geometry_outside_mask_immutable':geometry_drift==0,'mask_pixels':int((mask>0).sum())})
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);kept=0
 for cid in range(1,n):
  if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  kept+=1;component=lab==cid;row={'page':page,'component':cid,'method':'geometry_t80','candidate_threshold':GEOMETRY_THRESHOLD,'x':int(stats[cid,0]),'y':int(stats[cid,1])};row.update(surrounding_features(observed,component,routes));evidence_rows.append(row)
  t={'page':page,'component':cid,'method':'geometry_t80','candidate_threshold':GEOMETRY_THRESHOLD};t.update(truth_diagnostics(clean,component,routes));truth_rows.append(t)
 page_summary.append({'page':page,'components':kept,'mask_pixels':int((mask>0).sum()),'preserve_pixels':int(routes['preserve'].sum()),'remove_pixels':int(routes['remove'].sum()),'ambiguous_pixels':int(routes['ambiguous'].sum())})
 if index%10==0 or index==len(pages):print('Rebuilt',index,'/',len(pages),page)

cache_df=pd.DataFrame(cache_rows);evidence_df=pd.DataFrame(evidence_rows);truth_df=pd.DataFrame(truth_rows);integrity_df=pd.DataFrame(integrity_rows);page_df=pd.DataFrame(page_summary)
cache_df.to_csv(TABLES/'cache_status.csv',index=False);evidence_df.to_csv(TABLES/'surrounding_area_evidence_by_component.csv',index=False);truth_df.to_csv(TABLES/'surrounding_evidence_truth_diagnostics.csv',index=False);integrity_df.to_csv(TABLES/'telea_integrity.csv',index=False);page_df.to_csv(TABLES/'page_routing_summary.csv',index=False)
summary=evidence_df.groupby(['method','candidate_threshold'],as_index=False).agg(pages=('page','nunique'),components=('component','size'),mean_context_edge_density=('context_edge_density','mean'),mean_context_dark_density=('context_dark_density','mean'),mean_preserve_fraction=('preserve_fraction','mean'),mean_remove_fraction=('remove_fraction','mean'),mean_ambiguous_fraction=('ambiguous_fraction','mean'))
summary.to_csv(TABLES/'surrounding_area_evidence_summary.csv',index=False)


## Fail-closed compatibility checks and manifest


In [ ]:
required_cache={'mask','telea','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact'}
problems=[]
for row in cache_rows:
 with np.load(row['cache']) as z:
  missing=required_cache-set(z.files)
  if missing:problems.append({'page':row['page'],'problem':'missing_arrays','detail':'|'.join(sorted(missing))})
  else:
   mask=z['mask'];telea=z['telea'];geometry=z['geometry_t80'];observed=cv2.imread(str(observed_map[row['page']]),0)
   if not np.array_equal(telea[mask==0],observed[mask==0]):problems.append({'page':row['page'],'problem':'telea_outside_mask_drift','detail':''})
   if not np.array_equal(geometry[mask==0],observed[mask==0]):problems.append({'page':row['page'],'problem':'geometry_outside_mask_drift','detail':''})
if evidence_df.duplicated(['page','component']).any():problems.append({'page':'*','problem':'duplicate_component_rows','detail':''})
if len(evidence_df)!=len(truth_df):problems.append({'page':'*','problem':'evidence_truth_row_count_mismatch','detail':f'{len(evidence_df)} vs {len(truth_df)}'})
problems_df=pd.DataFrame(problems);problems_df.to_csv(TABLES/'compatibility_problems.csv',index=False)
verdict='PASS' if not problems else 'FAIL'
settings={'schema':'v7.5r-telea-immutable-reference-1','mask_dilation':MASK_DILATION,'crop_margin':CROP_MARGIN,'minimum_component_area':MIN_COMPONENT_AREA,'geometry_threshold':GEOMETRY_THRESHOLD,'band_width':BAND_WIDTH,'context_ring_radius':CONTEXT_RING_RADIUS}
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.5R','purpose':'corrected development reference for V7.8.2','run_path':str(OUT),'pages':len(pages),'components':len(evidence_df),'settings':settings,'settings_hash':object_hash(settings),'telea_outside_mask_immutable':bool(integrity_df.telea_outside_mask_immutable.all()),'geometry_outside_mask_immutable':bool(integrity_df.geometry_outside_mask_immutable.all()),'legacy_runs_modified':False,'independent_data_read':False,'compatibility_verdict':verdict,'next_required_step':'rebuild V7.7 structural calibration using this V7.5 reference'}
(OUT/'v7_5_rebuild_manifest.json').write_text(json.dumps(manifest,indent=2))
lines=['V7.5R CORRECTED DEVELOPMENT REFERENCE','',f'VERDICT: {verdict}',f'Run path: {OUT}',f'Pages: {len(pages)}',f'Components: {len(evidence_df)}',f'Telea outside-mask immutable: {integrity_df.telea_outside_mask_immutable.all()}',f'Geometry outside-mask immutable: {integrity_df.geometry_outside_mask_immutable.all()}',f'Compatibility problems: {len(problems)}','','NEXT: rebuild V7.7 structural calibration against this corrected V7.5 reference, then set V75_RUN_OVERRIDE and V77_RUN_OVERRIDE in V7.8.2.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines));print(chr(10).join(lines))
if problems:display(problems_df);raise RuntimeError('Corrected V7.5 reference failed compatibility checks.')
print('V75_RUN_OVERRIDE =',repr(str(OUT)))
